**`ingest_admin`**

Script examples to import administrative subdivisions for
new countries or country subdivisions

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from openplaces.api import get_admin1, get_admin2
from openplaces.core.schema import AdminId
from openplaces.io.ingest import get_recipe_data, ingest_recipe
from openplaces.recipe import get_recipe
from openplaces.timing import get_timer
from openplaces.utils import pretty_print

In [ ]:
REDO = False

In [ ]:
timer = get_timer('ingest_admin', verbose=True)

# Ingest data

In [ ]:
admin_id = AdminId('US')

## ``admin1``: states

In [ ]:
admin1_recipe = get_recipe('US', 'admin', source='admin1-uscensus-2024')
pretty_print(admin1_recipe)

In [ ]:
ingest_recipe(admin1_recipe, timer=timer, redo=True)

In [ ]:
admin1 = get_admin1(recipe=admin1_recipe)
admin1

## Counties

In [ ]:
admin2_recipe = get_recipe('US', 'admin', source='admin2-uscensus-2024')
pretty_print(admin2_recipe)

In [ ]:
ingest_recipe(admin2_recipe, timer=timer, redo=True)

In [ ]:
from openplaces.api import get_admin2
from openplaces.core.constants import STRING_SEPARATOR_WITHIN_IDS
from openplaces.utils import create_comparable_name_link

ADMIN2_JOIN_COLUMNS = ['admin1_id_leaf', 'name_link']

admin2_local = get_admin2(
    recipe=admin2_recipe,
    columns=['name', 'admin1_id_admin0', 'admin2_id_admin0', 'admin1_id_leaf'],
    # all_columns=True
)
admin2_local['name_link'] = admin2_local['name'].apply(create_comparable_name_link)

admin2 = get_admin2(admin_id)
admin2['admin1_id_leaf'] = admin2.index.str.split(STRING_SEPARATOR_WITHIN_IDS).map(
    lambda x: x[1]
)
admin2['name_link'] = admin2['name'].str.lower().apply(create_comparable_name_link)
admin2_local = admin2_local.join(
    admin2.reset_index().set_index(ADMIN2_JOIN_COLUMNS)['admin2_id'],
    on=ADMIN2_JOIN_COLUMNS,
)

# Show missing connections: official dataset
admin2_local[admin2_local['admin2_id'].isnull()].sort_values(ADMIN2_JOIN_COLUMNS)

In [ ]:
# Show missing connections: GADM
admin2[~admin2.index.isin(admin2_local['admin2_id'].dropna())].sort_values(ADMIN2_JOIN_COLUMNS)

## Towns
In progress

In [ ]:
from openplaces.recipe import get_recipe
recipe = get_recipe(admin_id, 'admin', source='admin3-uscensus-2024')
recipe

In [ ]:
from openplaces.io.ingest import get_recipe_data

gdf = get_recipe_data(recipe)
print(f'{len(gdf):,d} records.')
gdf.sample(5).T

In [ ]:
from openplaces.api import get_admin2
admin2 = get_admin2()

In [ ]:
admin2[admin2.index.str.startswith('US-MA')]